Why is it that using techniques like LoRa will not get the number of tokens down if I train it for a regional language?

In [1]:
# DISPLAY HELPERS ONLY — run this cell; no BPE logic is hidden here.
from html import escape
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None


def token_label(token_id, vocabulary):
    """Readable text when possible; exact hex bytes for partial UTF-8 characters."""
    piece = vocabulary[token_id]
    try:
        text = piece.decode("utf-8")
        text = text.replace(" ", "␠").replace("\n", "↵").replace("\t", "⇥")
        if not text.isprintable():
            raise ValueError("Show control bytes explicitly")
        return text
    except (UnicodeDecodeError, ValueError):
        return " ".join(f"0x{byte:02X}" for byte in piece)


def token_chip(token_id, vocabulary, show_ids=False, emphasized=False):
    colors = ["#e0f2fe", "#dcfce7", "#fef3c7", "#ede9fe", "#fce7f3", "#ccfbf1"]
    color = "#f1f5f9" if token_id < 256 else colors[(token_id - 256) % len(colors)]
    border = "2px solid #087f8c" if emphasized else "1px solid #cbd5e1"
    tooltip = escape(f"ID {token_id} | bytes {list(vocabulary[token_id])}", quote=True)
    label = escape(token_label(token_id, vocabulary))
    badge = f'<small style="color:#475569;font-size:10px">{token_id}</small>' if show_ids else ""
    return (f'<span title="{tooltip}" style="display:inline-flex;flex-direction:column;'
            f'align-items:center;justify-content:center;vertical-align:middle;background:{color};'
            f'border:{border};border-radius:7px;padding:5px 8px;margin:3px 2px;min-width:20px;'
            f'max-width:220px;overflow-wrap:anywhere;color:#0f172a;font:600 15px monospace">'
            f'{label}{badge}</span>')


def token_ribbon(token_ids, vocabulary, show_ids=False, pair=None, fresh_id=None):
    """Outline only the non-overlapping occurrences that merge_pair will replace."""
    parts = []
    position = 0
    limit = min(len(token_ids), 120)
    while position < limit:
        match = (pair is not None and position + 1 < limit
                 and (token_ids[position], token_ids[position + 1]) == pair)
        if match:
            left = token_chip(token_ids[position], vocabulary, show_ids)
            right = token_chip(token_ids[position + 1], vocabulary, show_ids)
            parts.append(f'<span style="display:inline-block;border:2px dashed #c07000;'
                         f'border-radius:9px;margin:2px;background:#fff8e5">{left}{right}</span>')
            position += 2
        else:
            token_id = token_ids[position]
            parts.append(token_chip(token_id, vocabulary, show_ids, token_id == fresh_id))
            position += 1
    if len(token_ids) > limit:
        parts.append(f'<p style="color:#64748b">Showing {limit} of {len(token_ids)} tokens; all are used.</p>')
    return '<div style="line-height:2;overflow-wrap:anywhere">' + ''.join(parts) + '</div>'


def html_table(headers, rows):
    # Callers supply already escaped labels or deliberately formatted HTML.
    head = ''.join(f'<th style="text-align:left;padding:7px 12px;border-bottom:2px solid #cbd5e1">{x}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td style="padding:6px 12px;border-bottom:1px solid #e2e8f0">{x}</td>' for x in row) + '</tr>' for row in rows)
    return f'<div style="overflow-x:auto"><table style="border-collapse:collapse;font-size:14px;color:#0f172a">{head}{body}</table></div>'


def panel(content):
    return ("<div style='font-family:system-ui,sans-serif;color:#0f172a;background:#fff;"
            "border:1px solid #cbd5e1;border-radius:12px;padding:20px;line-height:1.55;max-width:1050px'>"
            + content + "</div>")


def show_tokens(title, token_ids, vocabulary):
    display(HTML(panel(f'<b>{escape(title)}</b>' + token_ribbon(token_ids, vocabulary, show_ids=True))))

In [2]:
training_text = "banana bandana banana"
token_ids = list(training_text.encode("utf-8"))

# Every possible byte gets its own starting token.
vocabulary = {}
for byte_id in range(256):
    vocabulary[byte_id] = bytes([byte_id])

print("Training text:", repr(training_text))
print("Starting token IDs:", token_ids)
print("Tokens in this text:", len(token_ids))
show_tokens("Each chip is one token", token_ids, vocabulary)

Training text: 'banana bandana banana'
Starting token IDs: [98, 97, 110, 97, 110, 97, 32, 98, 97, 110, 100, 97, 110, 97, 32, 98, 97, 110, 97, 110, 97]
Tokens in this text: 21


### Count → choose → join → repeat

A **pair** means two neighboring tokens, in order. It does not mean two letters
anywhere in a word. After some merges, either token can already contain several letters.

Before running the next cells, look at the message and predict the most frequent pair.
You only need to count neighbors. BPE does not know what a banana is.

In [4]:
def count_pairs(token_ids):
    """Count every adjacent pair, including overlapping occurrences."""
    pair_counts = {}

    for position in range(len(token_ids) - 1):
        left_id = token_ids[position]
        right_id = token_ids[position + 1]
        pair = (left_id, right_id)

        if pair not in pair_counts:
            pair_counts[pair] = 0
        pair_counts[pair] += 1

    return pair_counts


def choose_pair(pair_counts):
    """Choose the highest count; ties go to the first pair encountered in the text."""
    best_pair = None
    best_count = 0

    for pair, count in pair_counts.items():
        if count > best_count:
            best_pair = pair
            best_count = count

    return best_pair

In [5]:
pair_counts = count_pairs(token_ids)
best_pair = choose_pair(pair_counts)

# Print readable labels so the numbers do not hide the idea.
print("PAIR          COUNT")
for pair, count in pair_counts.items():
    left = token_label(pair[0], vocabulary)
    right = token_label(pair[1], vocabulary)
    print(f"{left + ' + ' + right:<14}{count}")

print("\nWinner:", token_label(best_pair[0], vocabulary), "+",
      token_label(best_pair[1], vocabulary), "—", pair_counts[best_pair], "occurrences")

PAIR          COUNT
b + a         3
a + n         6
n + a         5
a + ␠         2
␠ + b         2
n + d         1
d + a         1

Winner: a + n — 6 occurrences


In [6]:
show_tokens("Each chip is one token", token_ids, vocabulary)

In [7]:
len(token_ids)

21

In [8]:
def merge_pair(token_ids, pair_to_merge, new_token_id):
    """Replace all non-overlapping matches of one pair, from left to right."""
    merged_ids = []
    position = 0

    while position < len(token_ids):
        has_neighbor = position + 1 < len(token_ids)

        if has_neighbor:
            current_pair = (token_ids[position], token_ids[position + 1])
        else:
            current_pair = None

        if current_pair == pair_to_merge:
            merged_ids.append(new_token_id)
            position += 2  # Both old tokens have been consumed.
        else:
            merged_ids.append(token_ids[position])
            position += 1

    return merged_ids

new_token_id = 256
left_id, right_id = best_pair
vocabulary[new_token_id] = vocabulary[left_id] + vocabulary[right_id]
after_one_merge = merge_pair(token_ids, best_pair, new_token_id)

print("New vocabulary entry:", new_token_id, "→", vocabulary[new_token_id])
print(f"Tokens: {len(token_ids)} → {len(after_one_merge)}")

display(HTML(panel(
    "<b>Before — dashed boxes show the pairs being joined</b>"
    + token_ribbon(token_ids, vocabulary, pair=best_pair)
    + "<p><b>After — every highlighted an is now one token</b></p>"
    + token_ribbon(after_one_merge, vocabulary, show_ids=True, fresh_id=new_token_id)
)))

New vocabulary entry: 256 → b'an'
Tokens: 21 → 15


In [9]:
def train_bpe(text, max_merges=30):
    """Learn repeated pairs from one small UTF-8 text."""
    token_ids = list(text.encode("utf-8"))
    vocabulary = {}
    for byte_id in range(256):
        vocabulary[byte_id] = bytes([byte_id])

    merge_rules = []

    for merge_number in range(max_merges):
        pair_counts = count_pairs(token_ids)
        best_pair = choose_pair(pair_counts)

        if best_pair is None:
            break  # There are fewer than two tokens left.
        if pair_counts[best_pair] < 2:
            break  # Classroom choice: learn only repeated pairs.

        new_token_id = 256 + merge_number
        left_id, right_id = best_pair
        vocabulary[new_token_id] = vocabulary[left_id] + vocabulary[right_id]

        token_ids = merge_pair(token_ids, best_pair, new_token_id)
        merge_rules.append((best_pair, new_token_id))

    return vocabulary, merge_rules

learned_vocabulary, learned_rules = train_bpe(training_text)
current_ids = list(training_text.encode("utf-8"))

rows = []
for step, (pair, new_id) in enumerate(learned_rules, start=1):
    before_count = len(current_ids)
    current_ids = merge_pair(current_ids, pair, new_id)
    left = token_label(pair[0], learned_vocabulary)
    right = token_label(pair[1], learned_vocabulary)
    joined = token_label(new_id, learned_vocabulary)
    rows.append([step, escape(f"{left} + {right} → {joined}"), new_id,
                 f"{before_count} → {len(current_ids)}"])

display(HTML(panel(html_table(["Step", "Rule learned", "New ID", "Tokens in text"], rows))))
show_tokens("The same text, with fewer tokens", current_ids, learned_vocabulary)
print("Vocabulary entries:", len(learned_vocabulary))

1,a + n → an,256,21 → 15
2,b + an → ban,257,15 → 12
3,an + a → ana,258,12 → 9
4,ban + ana → banana,259,9 → 7


Vocabulary entries: 260


In [13]:
def encode_text(text, merge_rules):
    """Use an already-trained tokenizer. This does not learn new rules."""
    token_ids = list(text.encode("utf-8"))

    for pair, new_token_id in merge_rules:
        token_ids = merge_pair(token_ids, pair, new_token_id)

    return token_ids


def decode_tokens(token_ids, vocabulary):
    """Rebuild the original bytes before decoding them as text."""
    all_bytes = b""
    for token_id in token_ids:
        all_bytes += vocabulary[token_id]

    return all_bytes.decode("utf-8")

In [14]:
# INTERFACE ONLY — the BPE algorithm is in the visible cells above.
PRESETS = {
    "Bananas · start here": "banana bandana banana",
    "Word pieces": "low lower lowest low lower",
    "Overlap puzzle": "aaaaaa",
    "Spaces are pieces too": "cat cat cat dog dog",
    "Hindi + emoji": "नमस्ते नमस्ते 🙂 नमस्ते",
}


def render_lab(text, vocabulary, rules, step, reveal, show_ids, new_text):
    available_rules = rules[:step]
    current_ids = encode_text(text, available_rules)
    initial_count = len(text.encode("utf-8"))
    reduction = 100 * (1 - len(current_ids) / initial_count) if initial_count else 0
    content = '<div style="color:#087f8c;font-size:12px;font-weight:700;letter-spacing:1px">BPE · MERGE LAB</div>'
    content += f'<h3 style="margin:6px 0 12px;color:#0f172a">{step} merges learned · {len(current_ids)} tokens now</h3>'
    content += (f'<p><b>{initial_count}</b> input bytes &nbsp; / &nbsp; '
                f'<b>{256 + step}</b> vocabulary entries &nbsp; / &nbsp; '
                f'<b>{reduction:.0f}%</b> fewer IDs than the byte sequence</p>')
    content += f'<p style="color:#475569">Loaded text: <code style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(repr(text))}</code></p>'

    fresh_id = None
    if step:
        pair, fresh_id = rules[step - 1]
        previous_ids = encode_text(text, rules[:step - 1])
        pair_count = count_pairs(previous_ids)[pair]
        replacements = len(previous_ids) - len(current_ids)
        left = escape(token_label(pair[0], vocabulary))
        right = escape(token_label(pair[1], vocabulary))
        joined = escape(token_label(fresh_id, vocabulary))
        content += (f'<div style="background:#f0fdfa;padding:12px;border-radius:8px">'
                    f'<b>Last merge: {left} + {right} → {joined}</b> · new ID {fresh_id}<br>'
                    f'{pair_count} adjacent occurrences; {replacements} non-overlapping replacements.<br>'
                    f'<b>Before:</b>{token_ribbon(previous_ids, vocabulary, show_ids, pair=pair)}</div>')

    next_pair = rules[step][0] if step < len(rules) else None
    content += '<p style="margin-bottom:3px"><b>Current token sequence</b></p>'
    content += token_ribbon(current_ids, vocabulary, show_ids,
                            pair=next_pair if reveal else None, fresh_id=fresh_id)
    content += '<p style="font-size:12px;color:#64748b">␠ space · ↵ newline · ⇥ tab · hover for exact bytes. Dashed boxes are mergeable pairs.</p>'

    if step == len(rules):
        counts = count_pairs(current_ids)
        best = choose_pair(counts)
        reason = ('the 30-merge budget was reached' if best is not None and counts[best] >= 2
                  else 'no adjacent pair appears at least twice')
        content += f'<p><b>Training stopped:</b> {reason}. Rewind or load another text.</p>'
    elif reveal:
        counts = count_pairs(current_ids)
        ordered_pairs = sorted(counts, key=counts.get, reverse=True)
        rows = []
        for rank, pair in enumerate(ordered_pairs[:8], start=1):
            left = escape(token_label(pair[0], vocabulary))
            right = escape(token_label(pair[1], vocabulary))
            rows.append([rank, f'{left} + {right}', counts[pair], 'Next merge' if pair == next_pair else ''])
        content += '<p><b>Next pair leaderboard</b> · highest count wins; ties keep first-encounter order.</p>'
        content += html_table(['Rank', 'Adjacent pair', 'Count', ''], rows)
        if len(ordered_pairs) > 8:
            content += f'<small>Top 8 of {len(ordered_pairs)} distinct pairs shown.</small>'
    else:
        content += '<p style="background:#fff8e5;padding:12px;border-radius:8px"><b>Predict:</b> which neighboring pair will be merged next? Then reveal the counts.</p>'

    if available_rules:
        rows = []
        for rank, (pair, new_id) in enumerate(available_rules, start=1):
            left = escape(token_label(pair[0], vocabulary))
            right = escape(token_label(pair[1], vocabulary))
            joined = escape(token_label(new_id, vocabulary))
            rows.append([rank, f'{left} + {right} → {joined}', new_id])
        content += '<details style="margin:15px 0"><summary style="cursor:pointer;font-weight:700">Inspect rules learned so far</summary>'
        content += html_table(['Order', 'Rule', 'ID'], rows) + '</details>'

    encoded = encode_text(new_text, available_rules)
    recovered = decode_tokens(encoded, vocabulary)
    content += '<hr style="border:0;border-top:1px solid #cbd5e1;margin:20px 0">'
    content += f'<b>New message · apply these {step} rules, without training</b>'
    content += f'<p><code style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(repr(new_text))}</code></p>'
    content += token_ribbon(encoded, vocabulary, show_ids)
    content += f'<p>{len(new_text.encode("utf-8"))} bytes → {len(encoded)} tokens &nbsp; · &nbsp; <b style="color:#087f8c">Exact round trip: {recovered == new_text}</b></p>'
    return panel(content)


def launch_lab():
    if widgets is None:
        print('Install ipywidgets using the setup cell, restart the kernel, then Run All for live controls.')
        display(HTML(render_lab(training_text, learned_vocabulary, learned_rules, 1, True, True, 'bandana banana!')))
        return None

    preset = widgets.Dropdown(options=list(PRESETS), description='Preset:',
                              layout=widgets.Layout(width='340px'))
    corpus = widgets.Textarea(value=PRESETS[preset.value], description='Train on:',
                              layout=widgets.Layout(width='98%', height='78px'))
    load_button = widgets.Button(description='Load text', button_style='info', icon='refresh')
    status = widgets.HTML()
    step = widgets.IntSlider(value=0, min=0, max=0, description='Merges:',
                             continuous_update=False, layout=widgets.Layout(width='95%'))
    next_button = widgets.Button(description='Merge next', button_style='success', icon='forward')
    undo_button = widgets.Button(description='Undo', icon='backward')
    rewind_button = widgets.Button(description='Rewind', icon='fast-backward')
    reveal = widgets.Checkbox(value=False, description='Reveal next pair', indent=False)
    show_ids = widgets.Checkbox(value=False, description='Show token IDs', indent=False)
    new_message = widgets.Text(value='bandana banana!', description='New message:',
                               continuous_update=False, style={'description_width': '110px'},
                               layout=widgets.Layout(width='98%'))
    board = widgets.HTML()
    state = {'busy': False}

    def redraw(change=None):
        if state['busy'] or 'rules' not in state:
            return
        next_button.disabled = step.value >= len(state['rules'])
        undo_button.disabled = rewind_button.disabled = step.value == 0
        if len(new_message.value) > 300:
            board.value = panel('Please shorten the new message to 300 characters or fewer.')
            return
        board.value = render_lab(state['text'], state['vocabulary'], state['rules'],
                                 step.value, reveal.value, show_ids.value, new_message.value)

    def load_text(button=None):
        if len(corpus.value) > 300:
            status.value = '<b style="color:#a33">Use at most 300 characters. The previous training text remains loaded.</b>'
            return
        state['busy'] = True
        state['text'] = corpus.value
        state['vocabulary'], state['rules'] = train_bpe(corpus.value)
        step.value = 0
        step.max = len(state['rules'])
        reveal.value = False
        state['busy'] = False
        status.value = '<span style="color:#087f8c">Text loaded. Start with a prediction, then reveal or merge.</span>'
        redraw()

    def choose_preset(change):
        corpus.value = PRESETS[change['new']]
        load_text()

    def mark_pending(change):
        if corpus.value != state.get('text'):
            status.value = '<span style="color:#a66300">Text edited. Click Load text to use it.</span>'

    def move_forward(button):
        step.value = min(step.max, step.value + 1)

    def move_back(button):
        step.value = max(0, step.value - 1)

    def rewind(button):
        step.value = 0

    load_button.on_click(load_text)
    next_button.on_click(move_forward)
    undo_button.on_click(move_back)
    rewind_button.on_click(rewind)
    preset.observe(choose_preset, names='value')
    corpus.observe(mark_pending, names='value')
    for control in (step, reveal, show_ids, new_message):
        control.observe(redraw, names='value')

    load_text()
    wrap = widgets.Layout(display='flex', flex_flow='row wrap', gap='8px')
    interface = widgets.VBox([
        preset, corpus, widgets.HBox([load_button, status], layout=wrap),
        step, widgets.HBox([next_button, undo_button, rewind_button], layout=wrap),
        widgets.HBox([reveal, show_ids], layout=wrap), new_message, board,
    ], layout=widgets.Layout(width='100%', max_width='1100px'))
    display(interface)
    return {'state': state, 'step': step, 'next': next_button, 'undo': undo_button,
            'rewind': rewind_button, 'preset': preset, 'corpus': corpus, 'load': load_button,
            'reveal': reveal, 'ids': show_ids, 'new_message': new_message, 'board': board,
            'status': status, 'interface': interface}

In [15]:
lab = launch_lab()